In [ ]:
#!/usr/bin/env python3
# ==========================================================
# MSHAT — Multi-Scale Hybrid Attention Transformer Refiner
# for Satellite Image Super-Resolution
# ==========================================================
#
# Pipeline:
#   Sentinel-2 LR (130×130, 10m)
#       → Frozen Tuned RCAN + HAT
#       → 520×520 intermediate SR (2.5m)
#       → MSHAT Transformer Refiner
#       → Final 520×520 4-band SR
#       → Compare against HR reference
#
# Final_SR = RCAN_HAT_SR + α × Transformer_Correction
#
# ==========================================================


In [ ]:
# ==========================================================
# CELL 1 — IMPORTS AND GPU CONFIGURATION
# ==========================================================

import os
import gc
import copy
import math
import time
import random
import warnings
from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

warnings.filterwarnings("ignore")

# ----- GPU setup -----
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = (
        torch.cuda.get_device_properties(0)
        .total_mem / 1024**3
    )
    print(f"GPU        : {gpu_name}")
    print(f"Memory     : {gpu_mem:.1f} GB")
    print(f"CUDA       : {torch.version.cuda}")
else:
    print("WARNING: No GPU detected. Training will be very slow.")

print(f"PyTorch    : {torch.__version__}")
print(f"Device     : {device}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

print(f"Seed       : {SEED}")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 2 — DATASET LOADING
# ==========================================================
# Uses tacoreader to load SEN2NAIPv2 from HuggingFace
# 100 training + 20 validation + 10 test images
# Bands: B02, B03, B04, B08 (4 channels)
# LR: 130×130 at 10m → HR: 520×520 at 2.5m
# Normalization: raw / 3000
# ==========================================================

import tacoreader

dataset = tacoreader.load(
    "tacofoundation:sen2naipv2"
)

# ----------------------------------------------------------
# Training pairs (indices 0–99)
# ----------------------------------------------------------

NUM_TRAIN = 100
train_pairs = []

for i in range(NUM_TRAIN):
    lr_path = dataset.read(i)["lr"]
    hr_path = dataset.read(i)["hr"]

    lr = np.load(lr_path).astype(np.float32) / 3000.0
    hr = np.load(hr_path).astype(np.float32) / 3000.0

    train_pairs.append((lr, hr))

    if (i + 1) % 25 == 0:
        print(f"  Loaded training {i+1}/{NUM_TRAIN}")

print(f"Training   : {len(train_pairs)} pairs")
print(f"  LR shape : {train_pairs[0][0].shape}")
print(f"  HR shape : {train_pairs[0][1].shape}")

# ----------------------------------------------------------
# Validation pairs (indices 100–119)
# ----------------------------------------------------------

NUM_VAL = 20
VAL_START = 100
val_pairs = []

for i in range(VAL_START, VAL_START + NUM_VAL):
    lr_path = dataset.read(i)["lr"]
    hr_path = dataset.read(i)["hr"]

    lr = np.load(lr_path).astype(np.float32) / 3000.0
    hr = np.load(hr_path).astype(np.float32) / 3000.0

    val_pairs.append((lr, hr))

print(f"Validation : {len(val_pairs)} pairs")

# ----------------------------------------------------------
# Test pairs (indices 150–159)  — NEVER used for training
# ----------------------------------------------------------

NUM_TEST = 10
TEST_START = 150
test_pairs = []

for i in range(TEST_START, TEST_START + NUM_TEST):
    lr_path = dataset.read(i)["lr"]
    hr_path = dataset.read(i)["hr"]

    lr = np.load(lr_path).astype(np.float32) / 3000.0
    hr = np.load(hr_path).astype(np.float32) / 3000.0

    test_pairs.append((lr, hr))

print(f"Test       : {len(test_pairs)} pairs")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 3 — RCAN + HAT ARCHITECTURE (FROZEN BASE MODEL)
# ==========================================================
# This is the Tuned RCAN + HAT that achieves:
#   RMSE  = 0.064346
#   PSNR  = 23.8295 dB
#   SSIM  = 0.712937
#
# Architecture:
#   16 RCAB blocks, 6 HAT blocks, 2 Reconstruction blocks
#   96 feature channels, ×4 PixelShuffle, 4-band output
# ==========================================================

class RCAB(nn.Module):
    """Residual Channel Attention Block."""

    def __init__(self, channels=96, reduction=8):
        super().__init__()

        self.conv1 = nn.Conv2d(
            channels, channels, 3, padding=1
        )
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(
            channels, channels, 3, padding=1
        )

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.attention = nn.Sequential(
            nn.Conv2d(
                channels, channels // reduction, 1
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                channels // reduction, channels, 1
            ),
            nn.Sigmoid()
        )

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.relu(out)
        out = self.conv2(out)

        weights = self.avg_pool(out)
        weights = self.attention(weights)
        out = out * weights

        return residual + 0.1 * out


class HATBlock(nn.Module):
    """Hybrid Attention Transformer Block."""

    def __init__(
        self,
        channels=96,
        num_heads=8,
        window_size=8,
        reduction=8
    ):
        super().__init__()

        self.window_size = window_size
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True
        )
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.channel_attention = nn.Sequential(
            nn.Conv2d(
                channels, channels // reduction, 1
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                channels // reduction, channels, 1
            ),
            nn.Sigmoid()
        )
        self.conv1 = nn.Conv2d(
            channels, channels * 2, 1
        )
        self.gelu = nn.GELU()
        self.conv2 = nn.Conv2d(
            channels * 2, channels, 1
        )

    def forward(self, x):
        B, C, H, W = x.shape
        ws = self.window_size

        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws

        if pad_h or pad_w:
            x_pad = F.pad(x, (0, pad_w, 0, pad_h))
        else:
            x_pad = x

        _, _, Hp, Wp = x_pad.shape

        windows = x_pad.permute(0, 2, 3, 1)
        windows = windows.reshape(
            B, Hp // ws, ws, Wp // ws, ws, C
        )
        windows = windows.permute(0, 1, 3, 2, 4, 5)
        windows = windows.reshape(-1, ws * ws, C)

        normalized = self.norm(windows)
        attended, _ = self.attention(
            normalized, normalized, normalized
        )
        windows = windows + attended

        windows = windows.reshape(
            B, Hp // ws, Wp // ws, ws, ws, C
        )
        windows = windows.permute(0, 1, 3, 2, 4, 5)
        out = windows.reshape(B, Hp, Wp, C)
        out = out.permute(0, 3, 1, 2)
        out = out[:, :, :H, :W]

        weights = self.avg_pool(out)
        weights = self.channel_attention(weights)
        out = out * weights

        residual = out
        out = self.conv1(out)
        out = self.gelu(out)
        out = self.conv2(out)
        out = out + residual

        return x + 0.1 * out


class ReconstructionBlock(nn.Module):
    """Post-attention reconstruction block."""

    def __init__(self, channels=96):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return x + 0.1 * self.body(x)


class RCAN_HAT_Final(nn.Module):
    """
    Tuned RCAN + HAT base model.
    16 RCAB + 6 HAT + 2 Reconstruction blocks.
    96 channels. ×4 PixelShuffle.
    """

    def __init__(
        self,
        num_rcab=16,
        num_hat=6,
        num_reconstruction=2,
        channels=96,
        scale=4
    ):
        super().__init__()

        self.head = nn.Conv2d(4, channels, 3, padding=1)

        self.rcan_body = nn.Sequential(
            *[RCAB(channels) for _ in range(num_rcab)]
        )
        self.rcan_conv = nn.Conv2d(
            channels, channels, 3, padding=1
        )

        self.hat_body = nn.Sequential(
            *[HATBlock(channels) for _ in range(num_hat)]
        )
        self.hat_conv = nn.Conv2d(
            channels, channels, 3, padding=1
        )

        self.reconstruction = nn.Sequential(
            *[
                ReconstructionBlock(channels)
                for _ in range(num_reconstruction)
            ]
        )
        self.reconstruction_conv = nn.Conv2d(
            channels, channels, 3, padding=1
        )

        self.up1 = nn.Sequential(
            nn.Conv2d(
                channels, channels * 4, 3, padding=1
            ),
            nn.PixelShuffle(2),
            nn.ReLU(inplace=True),
        )
        self.up2 = nn.Sequential(
            nn.Conv2d(
                channels, channels * 4, 3, padding=1
            ),
            nn.PixelShuffle(2),
            nn.ReLU(inplace=True),
        )

        self.tail = nn.Conv2d(channels, 4, 3, padding=1)

    def forward(self, x):
        shallow = self.head(x)

        x = self.rcan_body(shallow)
        x = self.rcan_conv(x)
        x = x + shallow

        hat_residual = x
        x = self.hat_body(x)
        x = self.hat_conv(x)
        x = x + hat_residual

        recon_residual = x
        x = self.reconstruction(x)
        x = self.reconstruction_conv(x)
        x = x + recon_residual

        x = self.up1(x)
        x = self.up2(x)
        x = self.tail(x)

        return x


# ----------------------------------------------------------
# Load and freeze the base model
# ----------------------------------------------------------

base_model = RCAN_HAT_Final(
    num_rcab=16,
    num_hat=6,
    num_reconstruction=2,
    channels=96,
    scale=4
).to(device)

# Load pre-trained weights
CHECKPOINT_PATH = (
    "/kaggle/input/rcan-hat-tuned/"
    "rcan_hat_reconstruction_tuned.pth"
)

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device,
        weights_only=True
    )
    base_model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    print("Loaded pre-trained RCAN+HAT weights")
    print(f"  Checkpoint RMSE: {checkpoint.get('RMSE', 'N/A')}")
    print(f"  Checkpoint PSNR: {checkpoint.get('PSNR', 'N/A')}")
    print(f"  Checkpoint SSIM: {checkpoint.get('SSIM', 'N/A')}")
else:
    print(f"WARNING: Checkpoint not found at {CHECKPOINT_PATH}")
    print("  The base model will use random weights.")
    print("  Update CHECKPOINT_PATH to your actual file.")

# Freeze all parameters
base_model.eval()
for param in base_model.parameters():
    param.requires_grad = False

num_base_params = sum(
    p.numel() for p in base_model.parameters()
)
print(f"\nRCAN+HAT frozen: {num_base_params:,} parameters")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 4 — EFFICIENT INTERMEDIATE-OUTPUT CACHING
# ==========================================================
# Pre-compute RCAN+HAT outputs for all training/val images.
# This eliminates running the frozen base model during
# every training step (~3× speedup).
# ==========================================================

print("Caching RCAN+HAT intermediate outputs...")
print("-" * 55)

def cache_base_model_outputs(pairs, name=""):
    """Run frozen RCAN+HAT on all images, return list of SR arrays."""
    cached_sr = []

    base_model.eval()

    with torch.no_grad():
        for i, (lr, hr) in enumerate(pairs):
            lr_tensor = torch.from_numpy(
                lr
            ).unsqueeze(0).float().to(device)

            sr = base_model(lr_tensor)
            sr = sr.squeeze(0).cpu().numpy()

            cached_sr.append(sr)

            if (i + 1) % 25 == 0 or (i + 1) == len(pairs):
                print(
                    f"  {name}: {i+1}/{len(pairs)} cached"
                )

    return cached_sr


# Cache training intermediates
train_sr_cached = cache_base_model_outputs(
    train_pairs, name="Train"
)

# Cache validation intermediates
val_sr_cached = cache_base_model_outputs(
    val_pairs, name="Val"
)

# Cache test intermediates (for final evaluation only)
test_sr_cached = cache_base_model_outputs(
    test_pairs, name="Test"
)

print(f"\nCached shapes:")
print(f"  Train SR : {len(train_sr_cached)} x {train_sr_cached[0].shape}")
print(f"  Val SR   : {len(val_sr_cached)} x {val_sr_cached[0].shape}")
print(f"  Test SR  : {len(test_sr_cached)} x {test_sr_cached[0].shape}")

# Free base model from GPU to save memory during training
# (we'll reload it for joint fine-tuning if needed)
base_model.cpu()
torch.cuda.empty_cache()
gc.collect()

print("\nBase model moved to CPU. GPU freed for refiner.")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 5 — SWIN TRANSFORMER BUILDING BLOCKS
# ==========================================================
# Window partition/reverse + Swin Transformer Block
# with regular and shifted window attention.
# ==========================================================

def window_partition(x, window_size):
    """
    Partition feature map into non-overlapping windows.

    Args:
        x: (B, C, H, W)
        window_size: int

    Returns:
        windows: (num_windows*B, window_size*window_size, C)
    """
    B, C, H, W = x.shape
    ws = window_size

    x = x.view(B, C, H // ws, ws, W // ws, ws)
    # (B, C, nH, ws, nW, ws) -> (B, nH, nW, ws, ws, C)
    x = x.permute(0, 2, 4, 3, 5, 1).contiguous()
    # (B*nH*nW, ws*ws, C)
    windows = x.view(-1, ws * ws, C)

    return windows


def window_reverse(windows, window_size, H, W, B):
    """
    Reverse window partition back to feature map.

    Args:
        windows: (num_windows*B, window_size*window_size, C)
        window_size: int
        H, W: original spatial dims
        B: batch size

    Returns:
        x: (B, C, H, W)
    """
    ws = window_size
    C = windows.shape[-1]

    x = windows.view(
        B, H // ws, W // ws, ws, ws, C
    )
    x = x.permute(0, 5, 1, 3, 2, 4).contiguous()
    x = x.view(B, C, H, W)

    return x


class WindowAttention(nn.Module):
    """
    Window-based Multi-Head Self-Attention with
    relative position bias (simplified).

    This is more powerful than nn.MultiheadAttention
    because it includes learnable relative position encoding.
    """

    def __init__(
        self,
        dim,
        window_size,
        num_heads,
        qkv_bias=True,
        attn_drop=0.0,
        proj_drop=0.0
    ):
        super().__init__()

        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        # QKV projection
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        # Relative position bias table
        # (2*ws-1) * (2*ws-1) possible relative positions
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros(
                (2 * window_size - 1) * (2 * window_size - 1),
                num_heads
            )
        )
        nn.init.trunc_normal_(
            self.relative_position_bias_table, std=0.02
        )

        # Compute relative position index
        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(
            torch.meshgrid(coords_h, coords_w, indexing="ij")
        )  # (2, ws, ws)
        coords_flatten = torch.flatten(coords, 1)  # (2, ws*ws)

        relative_coords = (
            coords_flatten[:, :, None]
            - coords_flatten[:, None, :]
        )  # (2, ws*ws, ws*ws)
        relative_coords = relative_coords.permute(
            1, 2, 0
        ).contiguous()  # (ws*ws, ws*ws, 2)
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1

        relative_position_index = relative_coords.sum(-1)
        self.register_buffer(
            "relative_position_index",
            relative_position_index
        )

    def forward(self, x, mask=None):
        """
        Args:
            x: (num_windows*B, N, C) where N = ws*ws
            mask: optional attention mask
        Returns:
            (num_windows*B, N, C)
        """
        B_, N, C = x.shape

        qkv = self.qkv(x).reshape(
            B_, N, 3, self.num_heads, self.head_dim
        ).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        # Scaled dot-product attention
        # Use float32 for attention computation (AMP stability)
        with torch.cuda.amp.autocast(enabled=False):
            q = q.float() * self.scale
            attn = q @ k.float().transpose(-2, -1)

            # Add relative position bias
            relative_position_bias = (
                self.relative_position_bias_table[
                    self.relative_position_index.view(-1)
                ].view(N, N, -1)
            )
            relative_position_bias = (
                relative_position_bias.permute(2, 0, 1)
                .contiguous()
                .float()
            )
            attn = attn + relative_position_bias.unsqueeze(0)

            # Apply mask for shifted windows
            if mask is not None:
                nW = mask.shape[0]
                attn = attn.view(
                    B_ // nW, nW,
                    self.num_heads, N, N
                )
                attn = attn + mask.unsqueeze(0).unsqueeze(2).float()
                attn = attn.view(-1, self.num_heads, N, N)

            attn = F.softmax(attn, dim=-1)

        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)

        return x


class SwinTransformerBlock(nn.Module):
    """
    Swin Transformer Block with optional shifted windows.

    Key differences from the existing WindowTransformerBlock:
    1. Relative position bias in attention
    2. Shifted window support (shift_size > 0)
    3. Proper pre-norm architecture
    4. Stronger MLP with dropout
    5. Learnable LayerScale instead of hard-coded 0.1 scaling
    """

    def __init__(
        self,
        dim,
        num_heads,
        window_size=8,
        shift_size=0,
        mlp_ratio=4.0,
        qkv_bias=True,
        drop=0.0,
        attn_drop=0.0,
        layer_scale_init=1e-2,
    ):
        super().__init__()

        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio

        # Attention
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim=dim,
            window_size=window_size,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
        )

        # MLP
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden, dim),
            nn.Dropout(drop),
        )

        # LayerScale: learnable per-channel scaling
        # Better than hard-coded 0.1
        self.gamma1 = nn.Parameter(
            layer_scale_init * torch.ones(dim)
        )
        self.gamma2 = nn.Parameter(
            layer_scale_init * torch.ones(dim)
        )

        # Cache for attention mask
        self._attn_mask = None
        self._mask_shape = None

    def _get_attn_mask(self, H, W, device):
        """Compute attention mask for shifted windows."""
        if self.shift_size == 0:
            return None

        # Check cache
        shape_key = (H, W)
        if self._attn_mask is not None and self._mask_shape == shape_key:
            return self._attn_mask

        ws = self.window_size
        ss = self.shift_size

        img_mask = torch.zeros(
            (1, 1, H, W), device=device
        )

        # Assign region indices for masking
        h_slices = (
            slice(0, -ws),
            slice(-ws, -ss),
            slice(-ss, None),
        )
        w_slices = (
            slice(0, -ws),
            slice(-ws, -ss),
            slice(-ss, None),
        )

        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, :, h, w] = cnt
                cnt += 1

        # Partition mask into windows
        mask_windows = window_partition(
            img_mask, ws
        )  # (nW, ws*ws, 1)
        mask_windows = mask_windows.squeeze(-1)  # (nW, ws*ws)

        attn_mask = (
            mask_windows.unsqueeze(1)
            - mask_windows.unsqueeze(2)
        )
        attn_mask = attn_mask.masked_fill(
            attn_mask != 0, float(-100.0)
        ).masked_fill(
            attn_mask == 0, float(0.0)
        )

        # Cache it
        self._attn_mask = attn_mask
        self._mask_shape = shape_key

        return attn_mask

    def forward(self, x):
        """
        Args:
            x: (B, C, H, W)
        Returns:
            (B, C, H, W)
        """
        B, C, H, W = x.shape
        ws = self.window_size
        ss = self.shift_size

        # Pad to multiples of window_size
        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws

        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")

        _, _, Hp, Wp = x.shape

        shortcut = x

        # --- Cyclic shift ---
        if ss > 0:
            shifted_x = torch.roll(
                x, shifts=(-ss, -ss), dims=(2, 3)
            )
            attn_mask = self._get_attn_mask(Hp, Wp, x.device)
        else:
            shifted_x = x
            attn_mask = None

        # --- Partition into windows ---
        windows = window_partition(
            shifted_x, ws
        )  # (nW*B, ws*ws, C)

        # --- Pre-norm + attention ---
        windows_normed = self.norm1(windows)
        attn_out = self.attn(windows_normed, mask=attn_mask)

        # LayerScale + residual
        windows = (
            windows
            + self.gamma1.unsqueeze(0).unsqueeze(0) * attn_out
        )

        # --- MLP ---
        windows = (
            windows
            + self.gamma2.unsqueeze(0).unsqueeze(0)
            * self.mlp(self.norm2(windows))
        )

        # --- Reverse windows ---
        x = window_reverse(windows, ws, Hp, Wp, B)

        # --- Reverse cyclic shift ---
        if ss > 0:
            x = torch.roll(
                x, shifts=(ss, ss), dims=(2, 3)
            )

        # Global residual
        x = shortcut + x

        # --- Remove padding ---
        if pad_h > 0 or pad_w > 0:
            x = x[:, :, :H + pad_h, :W + pad_w]
            # We keep padded size through the block and
            # let the caller handle final cropping
            # Actually, let's crop here for clean interface
            x = x[:, :, :H, :W]

            # Also crop shortcut if we use it
            # (shortcut is already padded, so this is fine)

        return x


class SwinBlockPair(nn.Module):
    """
    A pair of Swin Transformer blocks:
    1. W-MSA (regular window attention)
    2. SW-MSA (shifted window attention)

    This is the fundamental building unit of SwinIR/HAT.
    The shift enables cross-window information flow.
    """

    def __init__(
        self,
        dim,
        num_heads,
        window_size=8,
        mlp_ratio=4.0,
        drop=0.0,
        attn_drop=0.0,
    ):
        super().__init__()

        self.block1 = SwinTransformerBlock(
            dim=dim,
            num_heads=num_heads,
            window_size=window_size,
            shift_size=0,  # Regular windows
            mlp_ratio=mlp_ratio,
            drop=drop,
            attn_drop=attn_drop,
        )

        self.block2 = SwinTransformerBlock(
            dim=dim,
            num_heads=num_heads,
            window_size=window_size,
            shift_size=window_size // 2,  # Shifted windows
            mlp_ratio=mlp_ratio,
            drop=drop,
            attn_drop=attn_drop,
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return x

print("Swin Transformer blocks defined.")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 6 — CHANNEL ATTENTION BRIDGE (CAB)
# ==========================================================
# Squeeze-Excitation style channel attention.
# Learns inter-channel dependencies to preserve
# spectral consistency across B02/B03/B04/B08.
# ==========================================================

class ChannelAttentionBridge(nn.Module):
    """
    Channel Attention Bridge (CAB).

    Why: The 4-band multispectral data has critical inter-band
    relationships (e.g., NDVI). This module learns which feature
    channels should co-activate, enforcing spectral consistency
    in the feature space.

    Architecture:
        GlobalAvgPool -> FC(dim->dim//r) -> ReLU -> FC(dim//r->dim) -> Sigmoid -> Scale
    """

    def __init__(self, dim, reduction=4):
        super().__init__()

        self.body = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(dim, dim // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(dim // reduction, dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        weights = self.body(x).view(B, C, 1, 1)
        return x * weights


print("Channel Attention Bridge defined.")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 7 — SPECTRAL CROSS-ATTENTION MODULE
# ==========================================================
# Explicitly models inter-band correlations in feature space.
# Groups features by "band affinity" and applies cross-group
# attention so spectral relationships are architecturally
# enforced, not just loss-driven.
# ==========================================================

class SpectralCrossAttention(nn.Module):
    """
    Spectral Cross-Attention Module.

    Why: The 4-band output requires spectral coherence.
    The SAM loss provides gradients, but architectural
    enforcement is stronger. This module groups features
    into 4 spectral groups (one per output band) and
    applies cross-group attention.

    Architecture:
        Input: (B, C, H, W) with C=128
        -> Reshape to (B*H*W, 4, C//4) treating 4 groups as tokens
        -> Multi-head self-attention across the 4 groups
        -> Reshape back to (B, C, H, W)

    The attention learns which band-groups should co-vary
    (e.g., B04 and B08 for vegetation).
    """

    def __init__(self, dim, num_groups=4, num_heads=4):
        super().__init__()

        self.num_groups = num_groups
        self.group_dim = dim // num_groups

        self.norm = nn.LayerNorm(self.group_dim)
        self.attention = nn.MultiheadAttention(
            embed_dim=self.group_dim,
            num_heads=num_heads,
            batch_first=True,
        )

        self.proj = nn.Linear(
            self.group_dim, self.group_dim
        )

        # LayerScale
        self.gamma = nn.Parameter(
            0.01 * torch.ones(dim)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        G = self.num_groups
        Cg = self.group_dim

        shortcut = x

        # Reshape: (B, C, H, W) -> (B*H*W, G, Cg)
        x = x.view(B, G, Cg, H, W)
        x = x.permute(0, 3, 4, 1, 2).contiguous()
        x = x.view(B * H * W, G, Cg)

        # Cross-group attention
        normed = self.norm(x)
        attn_out, _ = self.attention(
            normed, normed, normed,
            need_weights=False,
        )
        x = x + attn_out

        # Project
        x = self.proj(x)

        # Reshape back: (B*H*W, G, Cg) -> (B, C, H, W)
        x = x.view(B, H, W, G, Cg)
        x = x.permute(0, 3, 4, 1, 2).contiguous()
        x = x.view(B, C, H, W)

        # LayerScale residual
        return shortcut + self.gamma.view(1, C, 1, 1) * x


print("Spectral Cross-Attention Module defined.")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 8 — MSHAT REFINER FULL ARCHITECTURE
# ==========================================================
# Multi-Scale Hybrid Attention Transformer Refiner
#
# Three-stage U-shaped architecture:
#   Stage 1: Full resolution (520x520), 128ch
#   Stage 2: Half resolution (260x260), 192ch
#   Stage 3: Full resolution (520x520), 128ch + skip
#
# Each stage: 2 SwinBlockPairs (= 4 Swin blocks)
#            + Channel Attention Bridge
#
# Stage 3 also includes Spectral Cross-Attention.
#
# Total: 12 Swin blocks + 3 CABs + 1 SpectralCrossAttn
# ==========================================================

class MultiScaleReconstructionHead(nn.Module):
    """
    Multi-scale feature aggregation for the final correction map.

    Why: Different error types have different spatial scales.
    - 1x1: per-pixel radiometric correction
    - 3x3: local edge refinement
    - 5x5: broader texture/area correction

    The three scales are fused and projected to the 4-band
    correction map.
    """

    def __init__(self, dim, out_channels=4):
        super().__init__()

        branch_dim = dim // 4

        self.branch_1x1 = nn.Sequential(
            nn.Conv2d(dim, branch_dim, 1),
            nn.GELU(),
        )
        self.branch_3x3 = nn.Sequential(
            nn.Conv2d(dim, branch_dim, 3, padding=1),
            nn.GELU(),
        )
        self.branch_5x5 = nn.Sequential(
            nn.Conv2d(dim, branch_dim, 5, padding=2),
            nn.GELU(),
        )

        # Fusion: 3*branch_dim -> dim
        self.fusion = nn.Sequential(
            nn.Conv2d(branch_dim * 3, dim, 1),
            nn.GELU(),
        )

        # Band-wise calibration
        self.calibration = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(dim // 2, out_channels, 3, padding=1),
        )

    def forward(self, x):
        b1 = self.branch_1x1(x)
        b3 = self.branch_3x3(x)
        b5 = self.branch_5x5(x)

        fused = self.fusion(
            torch.cat([b1, b3, b5], dim=1)
        )

        correction = self.calibration(fused)

        return correction


class MSHATRefiner(nn.Module):
    """
    Multi-Scale Hybrid Attention Transformer Refiner.

    Pipeline:
        RCAN+HAT SR (4ch, 520x520)
            -> Shallow Feature Extraction (128ch)
            -> Stage 1: Full-res refinement (4 Swin blocks)
            -> Downsample to half-res
            -> Stage 2: Context refinement (4 Swin blocks, 192ch)
            -> Upsample + skip from Stage 1
            -> Stage 3: Full-res fusion (4 Swin blocks)
            -> Spectral Cross-Attention
            -> Multi-Scale Reconstruction Head (-> 4ch correction)
            -> Output = SR_input + alpha * correction

    Parameters: ~8.5M
    """

    def __init__(
        self,
        in_channels=4,
        dim=128,
        dim_stage2=192,
        num_heads=8,
        window_size=8,
        num_pairs_per_stage=2,
        mlp_ratio=4.0,
        drop=0.0,
        attn_drop=0.0,
    ):
        super().__init__()

        self.window_size = window_size

        # ----- Shallow Feature Extraction -----
        self.head = nn.Sequential(
            nn.Conv2d(in_channels, dim, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(dim, dim, 3, padding=1),
        )

        # ----- Stage 1: Full resolution -----
        self.stage1 = nn.Sequential(
            *[
                SwinBlockPair(
                    dim=dim,
                    num_heads=num_heads,
                    window_size=window_size,
                    mlp_ratio=mlp_ratio,
                    drop=drop,
                    attn_drop=attn_drop,
                )
                for _ in range(num_pairs_per_stage)
            ]
        )
        self.stage1_conv = nn.Conv2d(
            dim, dim, 3, padding=1
        )
        self.cab1 = ChannelAttentionBridge(dim)

        # ----- Downsample: dim -> dim_stage2 -----
        self.downsample = nn.Sequential(
            nn.Conv2d(dim, dim_stage2, 4, stride=2, padding=1),
            nn.GELU(),
        )

        # ----- Stage 2: Half resolution -----
        self.stage2 = nn.Sequential(
            *[
                SwinBlockPair(
                    dim=dim_stage2,
                    num_heads=num_heads,
                    window_size=window_size,
                    mlp_ratio=mlp_ratio,
                    drop=drop,
                    attn_drop=attn_drop,
                )
                for _ in range(num_pairs_per_stage)
            ]
        )
        self.stage2_conv = nn.Conv2d(
            dim_stage2, dim_stage2, 3, padding=1
        )
        self.cab2 = ChannelAttentionBridge(dim_stage2)

        # ----- Upsample: dim_stage2 -> dim -----
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(
                dim_stage2, dim, 4, stride=2, padding=1
            ),
            nn.GELU(),
        )

        # ----- Skip fusion -----
        self.skip_fusion = nn.Conv2d(
            dim * 2, dim, 1
        )

        # ----- Stage 3: Full resolution with context -----
        self.stage3 = nn.Sequential(
            *[
                SwinBlockPair(
                    dim=dim,
                    num_heads=num_heads,
                    window_size=window_size,
                    mlp_ratio=mlp_ratio,
                    drop=drop,
                    attn_drop=attn_drop,
                )
                for _ in range(num_pairs_per_stage)
            ]
        )
        self.stage3_conv = nn.Conv2d(
            dim, dim, 3, padding=1
        )
        self.cab3 = ChannelAttentionBridge(dim)

        # ----- Spectral Cross-Attention -----
        self.spectral_attn = SpectralCrossAttention(
            dim=dim,
            num_groups=4,
            num_heads=4,
        )

        # ----- Reconstruction Head -----
        self.reconstruction = MultiScaleReconstructionHead(
            dim=dim,
            out_channels=in_channels,
        )

        # ----- Learnable residual scale -----
        # Initialized to 0.1 -- the network adjusts during training
        self.alpha = nn.Parameter(
            torch.tensor(0.1)
        )

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(
                m.weight, mode="fan_out", nonlinearity="relu"
            )
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, sr_input):
        """
        Args:
            sr_input: (B, 4, H, W) -- RCAN+HAT intermediate SR

        Returns:
            refined: (B, 4, H, W) -- corrected SR output
            correction: (B, 4, H, W) -- the correction map (for loss)
        """

        # Shallow features
        feat = self.head(sr_input)
        shallow = feat

        # ----- Stage 1 -----
        s1_residual = feat
        feat = self.stage1(feat)
        feat = self.stage1_conv(feat) + s1_residual
        feat = self.cab1(feat)
        s1_skip = feat  # Save for skip connection

        # ----- Downsample -----
        feat_down = self.downsample(feat)

        # ----- Stage 2 -----
        s2_residual = feat_down
        feat_down = self.stage2(feat_down)
        feat_down = self.stage2_conv(feat_down) + s2_residual
        feat_down = self.cab2(feat_down)

        # ----- Upsample -----
        feat_up = self.upsample(feat_down)

        # Handle size mismatch from stride-2 up/down
        _, _, H1, W1 = s1_skip.shape
        _, _, H2, W2 = feat_up.shape
        if H2 != H1 or W2 != W1:
            feat_up = F.interpolate(
                feat_up, size=(H1, W1),
                mode="bilinear", align_corners=False
            )

        # Skip fusion
        feat = self.skip_fusion(
            torch.cat([s1_skip, feat_up], dim=1)
        )

        # ----- Stage 3 -----
        s3_residual = feat
        feat = self.stage3(feat)
        feat = self.stage3_conv(feat) + s3_residual
        feat = self.cab3(feat)

        # ----- Spectral Cross-Attention -----
        feat = self.spectral_attn(feat)

        # Global residual with shallow features
        feat = feat + shallow

        # ----- Reconstruction -----
        correction = self.reconstruction(feat)

        # Residual refinement
        refined = sr_input + self.alpha * correction

        return refined, correction


print("MSHAT Refiner architecture defined.")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 9 — MODEL INSTANTIATION AND VERIFICATION
# ==========================================================

refiner = MSHATRefiner(
    in_channels=4,
    dim=128,
    dim_stage2=192,
    num_heads=8,
    window_size=8,
    num_pairs_per_stage=2,
    mlp_ratio=4.0,
    drop=0.0,
    attn_drop=0.0,
).to(device)

# Parameter count
num_params = sum(
    p.numel() for p in refiner.parameters()
)
trainable_params = sum(
    p.numel() for p in refiner.parameters()
    if p.requires_grad
)

print("MSHAT Transformer Refiner")
print("=" * 55)
print(f"Feature dim (Stage 1/3) : 128")
print(f"Feature dim (Stage 2)   : 192")
print(f"Attention heads         : 8")
print(f"Window size             : 8x8")
print(f"Swin block pairs/stage  : 2")
print(f"Total Swin blocks       : 12")
print(f"Channel Attention       : 3")
print(f"Spectral Cross-Attn     : 1")
print(f"Total parameters        : {num_params:,}")
print(f"Trainable parameters    : {trainable_params:,}")
print(f"Memory estimate (fp16)  : ~{num_params * 2 / 1024**2:.0f} MB")
print("=" * 55)

# Shape verification
print("\nShape verification:")
with torch.no_grad():
    dummy = torch.randn(1, 4, 192, 192).to(device)
    out, corr = refiner(dummy)
    print(f"  Input  : {dummy.shape}")
    print(f"  Output : {out.shape}")
    print(f"  Corr   : {corr.shape}")
    assert out.shape == dummy.shape, "Shape mismatch!"
    print("  Shape check passed")

    del dummy, out, corr
    torch.cuda.empty_cache()


In [ ]:
# ==========================================================
# CELL 10 — ALL 8 LOSS FUNCTIONS
# ==========================================================
# 1. Charbonnier       -- robust pixel reconstruction
# 2. MSE               -- directly optimizes PSNR
# 3. SSIM              -- structural similarity
# 4. Gradient/Edge     -- boundary preservation
# 5. Laplacian         -- anti-hallucination, texture
# 6. SAM               -- spectral angle consistency
# 7. Spectral Variance -- band statistics preservation
# 8. Residual L2       -- correction regularization
# ==========================================================

class CharbonnierLoss(nn.Module):
    """
    Charbonnier Loss: sqrt((x-y)^2 + eps^2)
    Smoother than L1 near zero, less sensitive to outliers than MSE.
    """
    def __init__(self, epsilon=1e-3):
        super().__init__()
        self.epsilon = epsilon

    def forward(self, pred, target):
        diff = pred - target
        return torch.mean(
            torch.sqrt(diff * diff + self.epsilon ** 2)
        )


class MSELoss(nn.Module):
    """
    Standard MSE Loss.
    Directly optimizes PSNR = 10*log10(1/MSE).
    This is the KEY addition over the previous loss function.
    """
    def forward(self, pred, target):
        return F.mse_loss(pred, target)


class SSIMLoss(nn.Module):
    """
    SSIM Loss: 1 - SSIM(pred, target)
    Computed per-band and averaged.

    Uses 11x11 Gaussian window as per the original SSIM paper
    (Wang et al., 2004).
    """
    def __init__(self, window_size=11, sigma=1.5):
        super().__init__()
        self.window_size = window_size

        # Create Gaussian kernel
        gauss = torch.Tensor([
            math.exp(
                -(x - window_size // 2) ** 2
                / (2 * sigma ** 2)
            )
            for x in range(window_size)
        ])
        gauss = gauss / gauss.sum()

        # 2D kernel from outer product
        kernel_2d = gauss.unsqueeze(1) * gauss.unsqueeze(0)
        kernel_2d = kernel_2d.unsqueeze(0).unsqueeze(0)

        self.register_buffer("kernel", kernel_2d)

        self.C1 = 0.01 ** 2
        self.C2 = 0.03 ** 2

    def _ssim_per_channel(self, pred, target):
        """Compute SSIM for a single channel."""
        ws = self.window_size
        kernel = self.kernel

        mu_p = F.conv2d(pred, kernel, padding=ws // 2)
        mu_t = F.conv2d(target, kernel, padding=ws // 2)

        mu_p_sq = mu_p * mu_p
        mu_t_sq = mu_t * mu_t
        mu_pt = mu_p * mu_t

        sigma_p_sq = (
            F.conv2d(pred * pred, kernel, padding=ws // 2)
            - mu_p_sq
        )
        sigma_t_sq = (
            F.conv2d(target * target, kernel, padding=ws // 2)
            - mu_t_sq
        )
        sigma_pt = (
            F.conv2d(pred * target, kernel, padding=ws // 2)
            - mu_pt
        )

        numerator = (
            (2 * mu_pt + self.C1)
            * (2 * sigma_pt + self.C2)
        )
        denominator = (
            (mu_p_sq + mu_t_sq + self.C1)
            * (sigma_p_sq + sigma_t_sq + self.C2)
        )

        ssim_map = numerator / denominator
        return ssim_map.mean()

    def forward(self, pred, target):
        """Compute 1-SSIM averaged over all bands."""
        B, C, H, W = pred.shape
        ssim_val = 0.0

        for c in range(C):
            ssim_val += self._ssim_per_channel(
                pred[:, c:c+1, :, :],
                target[:, c:c+1, :, :]
            )

        ssim_val = ssim_val / C

        return 1.0 - ssim_val


def gradient_loss(pred, target):
    """
    First-order gradient loss.
    Preserves edges: field boundaries, roads, buildings.
    """
    pred_dx = pred[:, :, :, 1:] - pred[:, :, :, :-1]
    pred_dy = pred[:, :, 1:, :] - pred[:, :, :-1, :]

    target_dx = target[:, :, :, 1:] - target[:, :, :, :-1]
    target_dy = target[:, :, 1:, :] - target[:, :, :-1, :]

    return (
        torch.mean(torch.abs(pred_dx - target_dx))
        + torch.mean(torch.abs(pred_dy - target_dy))
    )


class LaplacianLoss(nn.Module):
    """
    Laplacian (second-order) high-frequency loss.
    Penalizes hallucinated sharpening, which creates
    false second-derivative peaks.

    Kernel: [[0, 1, 0], [1, -4, 1], [0, 1, 0]]
    """
    def __init__(self):
        super().__init__()

        kernel = torch.tensor(
            [[0., 1., 0.],
             [1., -4., 1.],
             [0., 1., 0.]]
        ).unsqueeze(0).unsqueeze(0)

        self.register_buffer("kernel", kernel)

    def forward(self, pred, target):
        B, C, H, W = pred.shape
        loss = 0.0

        for c in range(C):
            lap_pred = F.conv2d(
                pred[:, c:c+1, :, :],
                self.kernel,
                padding=1
            )
            lap_target = F.conv2d(
                target[:, c:c+1, :, :],
                self.kernel,
                padding=1
            )
            loss += torch.mean(
                torch.abs(lap_pred - lap_target)
            )

        return loss / C


def spectral_angle_loss(pred, target):
    """
    Spectral Angle Mapper (SAM) loss.
    Preserves spectral relationships between B02/B03/B04/B08.
    """
    # Reshape to (N, 4) pixels
    pred_flat = pred.permute(0, 2, 3, 1).reshape(-1, 4)
    target_flat = target.permute(0, 2, 3, 1).reshape(-1, 4)

    pred_norm = F.normalize(pred_flat, p=2, dim=1, eps=1e-8)
    target_norm = F.normalize(target_flat, p=2, dim=1, eps=1e-8)

    cosine = torch.sum(pred_norm * target_norm, dim=1)
    cosine = torch.clamp(cosine, -1 + 1e-7, 1 - 1e-7)

    return torch.acos(cosine).mean()


def spectral_variance_loss(pred, target):
    """
    Spectral Variance Consistency loss.
    Prevents the refiner from collapsing band-specific variance.
    L = mean_over_bands( (var_pred_b - var_target_b)^2 )
    """
    # Per-band variance over spatial dimensions
    pred_var = pred.var(dim=(2, 3))     # (B, C)
    target_var = target.var(dim=(2, 3)) # (B, C)

    return torch.mean(
        (pred_var - target_var) ** 2
    )


def residual_l2_loss(correction):
    """
    L2 regularization on the correction magnitude.
    Prevents unnecessarily large corrections.
    """
    return torch.mean(correction ** 2)


print("All 8 loss functions defined.")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 11 — COMBINED LOSS WITH WEIGHTS
# ==========================================================

class CombinedLoss(nn.Module):
    """
    Multi-objective loss function combining all 8 components.

    Total = l1*Charb + l2*MSE + l3*SSIM + l4*Edge
          + l5*Laplacian + l6*SAM + l7*SpecVar + l8*ResL2
    """

    def __init__(self):
        super().__init__()

        self.charbonnier = CharbonnierLoss(epsilon=1e-3)
        self.mse_loss = MSELoss()
        self.ssim_loss = SSIMLoss(window_size=11, sigma=1.5)
        self.laplacian_loss = LaplacianLoss()

        # Weights
        self.w_charb = 1.0       # Primary pixel loss
        self.w_mse = 0.5         # CRITICAL: directly optimizes PSNR
        self.w_ssim = 0.15       # Structural preservation
        self.w_edge = 0.05       # Edge/boundary preservation
        self.w_laplacian = 0.05  # Anti-hallucination
        self.w_sam = 0.08        # Spectral consistency
        self.w_specvar = 0.03    # Band variance preservation
        self.w_residual = 0.02   # Correction regularization

    def forward(self, pred, target, correction):
        """
        Args:
            pred:       (B, 4, H, W) -- refined output
            target:     (B, 4, H, W) -- HR reference
            correction: (B, 4, H, W) -- correction map from refiner

        Returns:
            total_loss, loss_dict
        """
        losses = {}

        losses["charb"] = self.charbonnier(pred, target)
        losses["mse"] = self.mse_loss(pred, target)
        losses["ssim"] = self.ssim_loss(pred, target)
        losses["edge"] = gradient_loss(pred, target)
        losses["laplacian"] = self.laplacian_loss(pred, target)
        losses["sam"] = spectral_angle_loss(pred, target)
        losses["specvar"] = spectral_variance_loss(pred, target)
        losses["residual"] = residual_l2_loss(correction)

        total = (
            self.w_charb * losses["charb"]
            + self.w_mse * losses["mse"]
            + self.w_ssim * losses["ssim"]
            + self.w_edge * losses["edge"]
            + self.w_laplacian * losses["laplacian"]
            + self.w_sam * losses["sam"]
            + self.w_specvar * losses["specvar"]
            + self.w_residual * losses["residual"]
        )

        losses["total"] = total

        return total, losses


criterion = CombinedLoss().to(device)

print("Combined loss function created.")
print("-" * 55)
print("Loss weights:")
print(f"  Charbonnier     : {criterion.w_charb}")
print(f"  MSE             : {criterion.w_mse}")
print(f"  SSIM            : {criterion.w_ssim}")
print(f"  Edge/Gradient   : {criterion.w_edge}")
print(f"  Laplacian       : {criterion.w_laplacian}")
print(f"  SAM             : {criterion.w_sam}")
print(f"  Spectral Var    : {criterion.w_specvar}")
print(f"  Residual L2     : {criterion.w_residual}")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 12 — PATCH EXTRACTION + DATA AUGMENTATION
# ==========================================================
# Extract aligned patches from cached SR + HR pairs.
# Since both SR_cached and HR are at 520x520, we extract
# aligned patches at 192x192 from both.
# ==========================================================

PATCH_SIZE = 192  # HR patch size

def make_hr_patch(sr, hr, patch_size=192, augment=True):
    """
    Extract random aligned patches from SR and HR images.
    Both sr and hr are at 520x520 resolution.

    Args:
        sr: (4, 520, 520) -- cached RCAN+HAT output
        hr: (4, 520, 520) -- ground truth HR
        patch_size: size of patches to extract
        augment: apply random flips/rotations

    Returns:
        sr_patch: (4, patch_size, patch_size)
        hr_patch: (4, patch_size, patch_size)
    """
    _, H, W = sr.shape

    # Random top-left corner
    top = np.random.randint(0, H - patch_size + 1)
    left = np.random.randint(0, W - patch_size + 1)

    sr_patch = sr[:, top:top+patch_size, left:left+patch_size].copy()
    hr_patch = hr[:, top:top+patch_size, left:left+patch_size].copy()

    # Data augmentation (applied identically to both)
    if augment:
        # Random horizontal flip
        if random.random() > 0.5:
            sr_patch = sr_patch[:, :, ::-1].copy()
            hr_patch = hr_patch[:, :, ::-1].copy()

        # Random vertical flip
        if random.random() > 0.5:
            sr_patch = sr_patch[:, ::-1, :].copy()
            hr_patch = hr_patch[:, ::-1, :].copy()

        # Random 90 degree rotation
        k = random.randint(0, 3)
        if k > 0:
            sr_patch = np.rot90(
                sr_patch, k, axes=(1, 2)
            ).copy()
            hr_patch = np.rot90(
                hr_patch, k, axes=(1, 2)
            ).copy()

    return sr_patch, hr_patch


print(f"Patch extraction configured.")
print(f"  HR patch size : {PATCH_SIZE}x{PATCH_SIZE}")
print(f"  Augmentation  : flip + rotation")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 13 — OPTIMIZER, SCHEDULER, EMA
# ==========================================================

# ----- Optimizer -----
optimizer = torch.optim.AdamW(
    refiner.parameters(),
    lr=2e-4,
    weight_decay=1e-4,
    betas=(0.9, 0.999),
)

# ----- Scheduler -----
# CosineAnnealingWarmRestarts: smooth LR decay with restarts
EPOCHS = 50
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=EPOCHS,
    T_mult=1,
    eta_min=1e-6,
)

# ----- EMA (Exponential Moving Average) -----
# Provides smoother weights for evaluation.
# Usually gives 0.1-0.3 dB free PSNR improvement.


class EMA:
    """
    Exponential Moving Average of model parameters.

    EMA_weight = decay * EMA_weight + (1-decay) * model_weight

    At evaluation, we use EMA weights instead of raw
    training weights, which provides smoother, more
    generalizable predictions.
    """
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}

        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (
                    self.decay * self.shadow[name]
                    + (1.0 - self.decay) * param.data
                )

    def apply_shadow(self, model):
        """Replace model weights with EMA weights."""
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name].clone()

    def restore(self, model):
        """Restore original model weights."""
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name].clone()
        self.backup = {}


ema = EMA(refiner, decay=0.999)

print("Training configuration:")
print("-" * 55)
print(f"Optimizer     : AdamW")
print(f"Learning rate : 2e-4")
print(f"Weight decay  : 1e-4")
print(f"Scheduler     : CosineAnnealingWarmRestarts")
print(f"  T_0         : {EPOCHS}")
print(f"  eta_min     : 1e-6")
print(f"EMA decay     : 0.999")
print(f"Epochs        : {EPOCHS}")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 14 — AMP / MIXED PRECISION SETUP
# ==========================================================

scaler = GradScaler()

# Training hyperparameters
PATCHES_PER_EPOCH = 800
GRAD_CLIP = 1.0
VAL_EVERY = 5  # Validate every N epochs

print("Mixed precision training: ENABLED")
print(f"Patches/epoch : {PATCHES_PER_EPOCH}")
print(f"Grad clip     : {GRAD_CLIP}")
print(f"Val frequency : every {VAL_EVERY} epochs")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 15 — VALIDATION LOOP
# ==========================================================

def validate(model, val_sr_cached, val_pairs, device):
    """
    Run validation on full 520x520 images.
    Returns average PSNR, SSIM, and RMSE.
    """
    model.eval()

    all_psnr = []
    all_ssim = []
    all_rmse = []

    with torch.no_grad():
        for i, (sr_cached, (_, hr)) in enumerate(
            zip(val_sr_cached, val_pairs)
        ):
            sr_tensor = torch.from_numpy(
                sr_cached
            ).unsqueeze(0).float().to(device)

            # Full image inference
            refined, _ = model(sr_tensor)

            # Metrics on CPU numpy
            pred = refined.squeeze(0).cpu().numpy()
            target = hr

            # RMSE
            mse_val = np.mean((pred - target) ** 2)
            rmse_val = np.sqrt(mse_val)
            all_rmse.append(rmse_val)

            # PSNR
            psnr_val = 10 * np.log10(1.0 / (mse_val + 1e-10))
            all_psnr.append(psnr_val)

            # SSIM (per image)
            ssim_val = ssim(
                target.transpose(1, 2, 0),
                pred.transpose(1, 2, 0),
                channel_axis=2,
                data_range=1.0,
            )
            all_ssim.append(ssim_val)

    model.train()

    return {
        "psnr": np.mean(all_psnr),
        "ssim": np.mean(all_ssim),
        "rmse": np.mean(all_rmse),
    }


print("Validation loop defined.")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 16 — TRAINING LOOP
# ==========================================================

print("=" * 55)
print("TRAINING MSHAT TRANSFORMER REFINER")
print("=" * 55)

loss_history = []
val_history = []
best_val_psnr = -float("inf")
best_state = None
best_ema_state = None
best_epoch = -1

for epoch in range(EPOCHS):

    refiner.train()

    # Epoch accumulators
    epoch_losses = {
        "total": 0, "charb": 0, "mse": 0,
        "ssim": 0, "edge": 0, "laplacian": 0,
        "sam": 0, "specvar": 0, "residual": 0,
    }

    start_time = time.time()

    progress = tqdm(
        range(PATCHES_PER_EPOCH),
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        unit="patch",
        ncols=120,
    )

    for step in progress:

        # ------ Random training pair ------
        idx = np.random.randint(0, len(train_pairs))

        sr_cached = train_sr_cached[idx]
        _, hr = train_pairs[idx]

        # Random aligned patch with augmentation
        sr_patch, hr_patch = make_hr_patch(
            sr_cached, hr,
            patch_size=PATCH_SIZE,
            augment=True,
        )

        sr_tensor = torch.from_numpy(
            sr_patch
        ).unsqueeze(0).float().to(device)

        hr_tensor = torch.from_numpy(
            hr_patch
        ).unsqueeze(0).float().to(device)

        # ------ Forward pass (AMP) ------
        optimizer.zero_grad(set_to_none=True)

        with autocast():
            refined, correction = refiner(sr_tensor)
            loss, loss_dict = criterion(
                refined, hr_tensor, correction
            )

        # ------ Backward pass ------
        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            refiner.parameters(), GRAD_CLIP
        )

        scaler.step(optimizer)
        scaler.update()

        # ------ EMA update ------
        ema.update(refiner)

        # ------ Accumulate losses ------
        for key in epoch_losses:
            if key in loss_dict:
                epoch_losses[key] += loss_dict[key].item()

        # Progress bar
        progress.set_postfix(
            loss=f"{epoch_losses['total']/(step+1):.4f}",
            charb=f"{epoch_losses['charb']/(step+1):.4f}",
            mse=f"{epoch_losses['mse']/(step+1):.5f}",
            alpha=f"{refiner.alpha.item():.4f}",
        )

    # ------ Scheduler step ------
    scheduler.step()

    # ------ Epoch summary ------
    elapsed = time.time() - start_time

    avg_losses = {
        k: v / PATCHES_PER_EPOCH
        for k, v in epoch_losses.items()
    }
    loss_history.append(avg_losses)

    print(f"\n{'='*55}")
    print(f"Epoch {epoch+1}/{EPOCHS} -- {elapsed:.1f}s")
    print(f"  Total     : {avg_losses['total']:.6f}")
    print(f"  Charb     : {avg_losses['charb']:.6f}")
    print(f"  MSE       : {avg_losses['mse']:.6f}")
    print(f"  SSIM      : {avg_losses['ssim']:.6f}")
    print(f"  Edge      : {avg_losses['edge']:.6f}")
    print(f"  Laplacian : {avg_losses['laplacian']:.6f}")
    print(f"  SAM       : {avg_losses['sam']:.6f}")
    print(f"  SpecVar   : {avg_losses['specvar']:.8f}")
    print(f"  ResL2     : {avg_losses['residual']:.8f}")
    print(f"  Alpha     : {refiner.alpha.item():.4f}")
    print(f"  LR        : {optimizer.param_groups[0]['lr']:.2e}")

    # ------ Validation ------
    if (epoch + 1) % VAL_EVERY == 0 or (epoch + 1) == EPOCHS:

        # Validate with EMA weights
        ema.apply_shadow(refiner)

        val_metrics = validate(
            refiner, val_sr_cached, val_pairs, device
        )

        ema.restore(refiner)

        val_history.append({
            "epoch": epoch + 1,
            "psnr": val_metrics["psnr"],
            "ssim": val_metrics["ssim"],
            "rmse": val_metrics["rmse"],
        })

        print(f"\n  Validation (EMA):")
        print(f"    PSNR : {val_metrics['psnr']:.4f} dB")
        print(f"    SSIM : {val_metrics['ssim']:.4f}")
        print(f"    RMSE : {val_metrics['rmse']:.6f}")

        # ------ Best checkpoint selection ------
        if val_metrics["psnr"] > best_val_psnr:
            best_val_psnr = val_metrics["psnr"]
            best_epoch = epoch + 1

            # Save EMA weights as best
            ema.apply_shadow(refiner)
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in refiner.state_dict().items()
            }
            ema.restore(refiner)

            print(f"    * New best! PSNR={best_val_psnr:.4f} dB")

    print(f"{'='*55}")


print(f"\nTraining complete.")
print(f"Best validation PSNR: {best_val_psnr:.4f} dB (epoch {best_epoch})")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 17 — RESTORE BEST CHECKPOINT
# ==========================================================

if best_state is not None:
    refiner.load_state_dict(best_state)
    refiner = refiner.to(device)
    print(f"Best model restored (epoch {best_epoch})")
    print(f"Validation PSNR: {best_val_psnr:.4f} dB")
else:
    print("WARNING: No best state found. Using final weights.")

print("=" * 55)


In [ ]:
# ==========================================================
# CELL 18 — TEST INFERENCE
# ==========================================================
# Run the full pipeline on the 10 unseen test images.
# These were NEVER used for training or model selection.
# ==========================================================

print("Running test inference...")
print("-" * 55)

refiner.eval()

final_predictions = []
final_targets = []
base_predictions = []

with torch.no_grad():
    for i, ((_, hr), sr_cached) in enumerate(
        zip(test_pairs, test_sr_cached)
    ):
        sr_tensor = torch.from_numpy(
            sr_cached
        ).unsqueeze(0).float().to(device)

        # Transformer refinement
        refined, correction = refiner(sr_tensor)

        # Store results
        pred = refined.squeeze(0).cpu().numpy()
        final_predictions.append(pred)
        final_targets.append(hr)
        base_predictions.append(sr_cached)

        print(f"  Test {TEST_START+i}: done")

final_predictions = np.array(final_predictions)
final_targets = np.array(final_targets)
base_predictions = np.array(base_predictions)

print(f"\nPredictions shape: {final_predictions.shape}")
print(f"Targets shape    : {final_targets.shape}")
print(f"Base pred shape  : {base_predictions.shape}")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 19 — AGGREGATE METRICS
# ==========================================================

def compute_metrics(predictions, targets):
    """Compute RMSE, PSNR, SSIM over all images."""

    mse = np.mean((predictions - targets) ** 2)
    rmse = np.sqrt(mse)
    psnr = 10 * np.log10(1.0 / (mse + 1e-10))

    ssim_scores = []
    for i in range(len(targets)):
        score = ssim(
            targets[i].transpose(1, 2, 0),
            predictions[i].transpose(1, 2, 0),
            channel_axis=2,
            data_range=1.0,
        )
        ssim_scores.append(score)

    mean_ssim = np.mean(ssim_scores)

    return {
        "rmse": rmse,
        "psnr": psnr,
        "ssim": mean_ssim,
        "ssim_per_image": ssim_scores,
    }


# ----- RCAN + HAT baseline -----
base_metrics = compute_metrics(
    base_predictions, final_targets
)

# ----- RCAN + HAT + MSHAT Refiner -----
refined_metrics = compute_metrics(
    final_predictions, final_targets
)

print("=" * 55)
print("TEST RESULTS")
print("=" * 55)

print("\nRCAN + HAT (baseline):")
print(f"  RMSE : {base_metrics['rmse']:.6f}")
print(f"  PSNR : {base_metrics['psnr']:.4f} dB")
print(f"  SSIM : {base_metrics['ssim']:.4f}")

print("\nRCAN + HAT + MSHAT Refiner:")
print(f"  RMSE : {refined_metrics['rmse']:.6f}")
print(f"  PSNR : {refined_metrics['psnr']:.4f} dB")
print(f"  SSIM : {refined_metrics['ssim']:.4f}")

print("\nImprovement:")
print(f"  dRMSE : {refined_metrics['rmse'] - base_metrics['rmse']:.6f}")
print(f"  dPSNR : {refined_metrics['psnr'] - base_metrics['psnr']:+.4f} dB")
print(f"  dSSIM : {refined_metrics['ssim'] - base_metrics['ssim']:+.4f}")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 20 — PER-IMAGE METRICS
# ==========================================================

print("=" * 55)
print("PER-IMAGE TEST METRICS")
print("=" * 55)

print(f"\n{'Image':<10} {'RMSE_base':>10} {'RMSE_ref':>10} "
      f"{'PSNR_base':>10} {'PSNR_ref':>10} "
      f"{'SSIM_base':>10} {'SSIM_ref':>10}")
print("-" * 75)

for i in range(len(final_targets)):
    # Base metrics per image
    mse_b = np.mean(
        (base_predictions[i] - final_targets[i]) ** 2
    )
    rmse_b = np.sqrt(mse_b)
    psnr_b = 10 * np.log10(1.0 / (mse_b + 1e-10))
    ssim_b = ssim(
        final_targets[i].transpose(1, 2, 0),
        base_predictions[i].transpose(1, 2, 0),
        channel_axis=2, data_range=1.0,
    )

    # Refined metrics per image
    mse_r = np.mean(
        (final_predictions[i] - final_targets[i]) ** 2
    )
    rmse_r = np.sqrt(mse_r)
    psnr_r = 10 * np.log10(1.0 / (mse_r + 1e-10))
    ssim_r = ssim(
        final_targets[i].transpose(1, 2, 0),
        final_predictions[i].transpose(1, 2, 0),
        channel_axis=2, data_range=1.0,
    )

    print(
        f"Test {TEST_START+i:<4} "
        f"{rmse_b:>10.6f} {rmse_r:>10.6f} "
        f"{psnr_b:>10.4f} {psnr_r:>10.4f} "
        f"{ssim_b:>10.4f} {ssim_r:>10.4f}"
    )

print("=" * 55)


In [ ]:
# ==========================================================
# CELL 21 — COMPARISON TABLE (ALL MODELS)
# ==========================================================

import pandas as pd

results = pd.DataFrame({
    "Model": [
        "RCAN Baseline",
        "Improved RCAN",
        "RCAN + HAT",
        "RCAN + HAT v2",
        "RCAN + SwinIR",
        "SwinIR Only",
        "Tuned RCAN",
        "Tuned RCAN + HAT",
        "Tuned RCAN + HAT + MSHAT Refiner",
    ],
    "RMSE": [
        0.05746,
        0.05548,
        0.06555,
        0.06546,
        0.06577,
        0.06721,
        0.06735,
        base_metrics["rmse"],
        refined_metrics["rmse"],
    ],
    "PSNR": [
        24.813,
        25.118,
        23.669,
        23.680,
        23.639,
        23.452,
        23.433,
        base_metrics["psnr"],
        refined_metrics["psnr"],
    ],
    "SSIM": [
        0.6793,
        0.6905,
        0.7048,
        0.7061,
        np.nan,
        0.6958,
        0.7095,
        base_metrics["ssim"],
        refined_metrics["ssim"],
    ],
})

print("=" * 55)
print("ALL MODEL COMPARISON")
print("=" * 55)

display(
    results.sort_values(
        "PSNR", ascending=False
    ).reset_index(drop=True)
)


In [ ]:
# ==========================================================
# CELL 22 — VISUAL COMPARISON
# ==========================================================
# Side-by-side: LR | Bicubic | RCAN+HAT | MSHAT Refined | HR
# ==========================================================

def stretch_rgb(img):
    """
    Convert 4-band satellite image to displayable RGB.
    Uses bands B04 (Red), B03 (Green), B02 (Blue).
    """
    if img.ndim == 3 and img.shape[0] <= 4:
        # (C, H, W) -> (H, W, C)
        img = img.transpose(1, 2, 0)

    # Take RGB bands (B04=idx2, B03=idx1, B02=idx0)
    rgb = img[:, :, [2, 1, 0]]

    # Contrast stretch
    p2, p98 = np.percentile(rgb, (2, 98))
    rgb = np.clip(
        (rgb - p2) / (p98 - p2 + 1e-8),
        0, 1
    )

    return rgb


n_display = min(5, len(test_pairs))

fig, axes = plt.subplots(
    n_display, 5,
    figsize=(25, 5 * n_display),
)

if n_display == 1:
    axes = axes[np.newaxis, :]

for i in range(n_display):

    lr = test_pairs[i][0]
    hr = final_targets[i]
    base_sr = base_predictions[i]
    refined_sr = final_predictions[i]

    # Bicubic upscale
    lr_tensor = torch.from_numpy(
        lr
    ).unsqueeze(0).float()
    bicubic = F.interpolate(
        lr_tensor, size=(520, 520),
        mode="bicubic", align_corners=False,
    ).squeeze(0).numpy()

    # --- Column 0: LR ---
    axes[i, 0].imshow(stretch_rgb(lr))
    axes[i, 0].set_title(
        f"Test {TEST_START+i} -- LR 10m",
        fontsize=10,
    )
    axes[i, 0].axis("off")

    # --- Column 1: Bicubic ---
    axes[i, 1].imshow(stretch_rgb(bicubic))
    axes[i, 1].set_title("Bicubic x4", fontsize=10)
    axes[i, 1].axis("off")

    # --- Column 2: RCAN + HAT ---
    axes[i, 2].imshow(stretch_rgb(base_sr))
    axes[i, 2].set_title(
        f"RCAN+HAT",
        fontsize=10,
    )
    axes[i, 2].axis("off")

    # --- Column 3: MSHAT Refined ---
    axes[i, 3].imshow(stretch_rgb(refined_sr))

    # Per-image metrics for title
    mse_r = np.mean(
        (refined_sr - hr) ** 2
    )
    psnr_r = 10 * np.log10(1.0 / (mse_r + 1e-10))

    axes[i, 3].set_title(
        f"MSHAT Refined (PSNR={psnr_r:.2f})",
        fontsize=10,
    )
    axes[i, 3].axis("off")

    # --- Column 4: HR Reference ---
    axes[i, 4].imshow(stretch_rgb(hr))
    axes[i, 4].set_title(
        "HR Reference 2.5m",
        fontsize=10,
    )
    axes[i, 4].axis("off")

plt.suptitle(
    "LR -> Bicubic -> RCAN+HAT -> MSHAT Refined -> HR Reference",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.savefig(
    "/kaggle/working/mshat_comparison.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

print("Visual comparison saved.")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 22b — CORRECTION MAP VISUALIZATION
# ==========================================================
# Visualize what the Transformer is actually correcting.
# ==========================================================

n_display = min(3, len(test_pairs))

fig, axes = plt.subplots(
    n_display, 4,
    figsize=(20, 5 * n_display),
)

if n_display == 1:
    axes = axes[np.newaxis, :]

refiner.eval()

with torch.no_grad():
    for i in range(n_display):
        sr_tensor = torch.from_numpy(
            test_sr_cached[i]
        ).unsqueeze(0).float().to(device)

        _, correction = refiner(sr_tensor)
        corr = correction.squeeze(0).cpu().numpy()

        # Per-band correction magnitude
        for b, band_name in enumerate(
            ["B02 (Blue)", "B03 (Green)", "B04 (Red)", "B08 (NIR)"]
        ):
            if b < 4:
                ax = axes[i, b]
                corr_band = corr[b]

                # Symmetric colormap centered at 0
                vmax = max(
                    abs(corr_band.min()),
                    abs(corr_band.max())
                )
                vmax = max(vmax, 0.001)

                im = ax.imshow(
                    corr_band,
                    cmap="RdBu_r",
                    vmin=-vmax,
                    vmax=vmax,
                )
                ax.set_title(
                    f"Test {TEST_START+i} -- {band_name}\n"
                    f"alpha={refiner.alpha.item():.3f}, "
                    f"range=[{corr_band.min():.4f}, {corr_band.max():.4f}]",
                    fontsize=9,
                )
                ax.axis("off")
                plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle(
    "Correction Maps by Band -- What the Transformer Corrects",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(
    "/kaggle/working/mshat_corrections.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

print("Correction maps saved.")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 23 — SAVE COMPLETE PIPELINE
# ==========================================================

save_dir = "/kaggle/working/"
os.makedirs(save_dir, exist_ok=True)

# ----- Save refiner -----
refiner_save_path = os.path.join(
    save_dir, "mshat_refiner.pth"
)

refiner_checkpoint = {
    "model_name": "MSHAT Transformer Refiner",
    "model_state_dict": refiner.state_dict(),

    # Architecture config
    "config": {
        "in_channels": 4,
        "dim": 128,
        "dim_stage2": 192,
        "num_heads": 8,
        "window_size": 8,
        "num_pairs_per_stage": 2,
        "mlp_ratio": 4.0,
    },

    # Training info
    "epochs": EPOCHS,
    "best_epoch": best_epoch,
    "best_val_psnr": best_val_psnr,
    "patch_size": PATCH_SIZE,
    "patches_per_epoch": PATCHES_PER_EPOCH,

    # Loss weights
    "loss_weights": {
        "charbonnier": criterion.w_charb,
        "mse": criterion.w_mse,
        "ssim": criterion.w_ssim,
        "edge": criterion.w_edge,
        "laplacian": criterion.w_laplacian,
        "sam": criterion.w_sam,
        "specvar": criterion.w_specvar,
        "residual": criterion.w_residual,
    },

    # Results
    "test_metrics": {
        "rmse": float(refined_metrics["rmse"]),
        "psnr": float(refined_metrics["psnr"]),
        "ssim": float(refined_metrics["ssim"]),
    },

    "base_metrics": {
        "rmse": float(base_metrics["rmse"]),
        "psnr": float(base_metrics["psnr"]),
        "ssim": float(base_metrics["ssim"]),
    },

    # Alpha value
    "alpha": float(refiner.alpha.item()),

    # Normalization
    "normalization": 3000.0,

    # Training history
    "loss_history": loss_history,
    "val_history": val_history,
}

torch.save(refiner_checkpoint, refiner_save_path)

print(f"Refiner saved: {refiner_save_path}")
print(
    f"  File size: "
    f"{os.path.getsize(refiner_save_path) / 1024**2:.1f} MB"
)

# ----- Save full pipeline config -----
pipeline_config_path = os.path.join(
    save_dir, "pipeline_config.pth"
)

pipeline_config = {
    "pipeline": "RCAN_HAT + MSHAT_Refiner",

    "base_model": {
        "class": "RCAN_HAT_Final",
        "num_rcab": 16,
        "num_hat": 6,
        "num_reconstruction": 2,
        "channels": 96,
        "scale": 4,
        "checkpoint": "rcan_hat_reconstruction_tuned.pth",
    },

    "refiner": {
        "class": "MSHATRefiner",
        "in_channels": 4,
        "dim": 128,
        "dim_stage2": 192,
        "num_heads": 8,
        "window_size": 8,
        "num_pairs_per_stage": 2,
        "mlp_ratio": 4.0,
        "checkpoint": "mshat_refiner.pth",
    },

    "dataset": {
        "name": "SEN2NAIPv2",
        "bands": ["B02", "B03", "B04", "B08"],
        "lr_size": [130, 130],
        "hr_size": [520, 520],
        "scale": 4,
        "normalization": 3000.0,
    },
}

torch.save(pipeline_config, pipeline_config_path)

print(f"Pipeline config saved: {pipeline_config_path}")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 24 — RELOAD PIPELINE FOR INFERENCE
# ==========================================================
# This cell shows how to reload the complete pipeline
# from saved checkpoints and run inference on new data.
# ==========================================================

def load_pipeline(
    base_checkpoint_path,
    refiner_checkpoint_path,
    device="cuda",
):
    """
    Load the complete SR pipeline from saved checkpoints.

    Returns:
        base_model: frozen RCAN+HAT
        refiner: MSHAT Transformer Refiner
    """
    # ----- Load base model -----
    loaded_base_model = RCAN_HAT_Final(
        num_rcab=16,
        num_hat=6,
        num_reconstruction=2,
        channels=96,
        scale=4,
    ).to(device)

    base_ckpt = torch.load(
        base_checkpoint_path,
        map_location=device,
        weights_only=True,
    )
    loaded_base_model.load_state_dict(
        base_ckpt["model_state_dict"]
    )
    loaded_base_model.eval()

    for p in loaded_base_model.parameters():
        p.requires_grad = False

    # ----- Load refiner -----
    refiner_ckpt = torch.load(
        refiner_checkpoint_path,
        map_location=device,
        weights_only=True,
    )

    config = refiner_ckpt["config"]
    loaded_refiner = MSHATRefiner(**config).to(device)
    loaded_refiner.load_state_dict(
        refiner_ckpt["model_state_dict"]
    )
    loaded_refiner.eval()

    print(f"Pipeline loaded successfully.")
    print(f"  Base model : RCAN_HAT_Final")
    print(f"  Refiner    : MSHATRefiner")
    print(f"  Alpha      : {loaded_refiner.alpha.item():.4f}")
    print(f"  Best PSNR  : {refiner_ckpt.get('best_val_psnr', 'N/A')}")

    return loaded_base_model, loaded_refiner


def run_inference(
    inference_base_model,
    inference_refiner,
    lr_image,
    device="cuda",
    normalization=3000.0,
):
    """
    Run the complete SR pipeline on a single LR image.

    Args:
        inference_base_model: frozen RCAN+HAT
        inference_refiner: MSHAT Transformer Refiner
        lr_image: numpy array (4, 130, 130) -- raw or normalized
        device: 'cuda' or 'cpu'
        normalization: divide by this if raw reflectance

    Returns:
        sr_output: numpy array (4, 520, 520) -- normalized [0, 1]
    """
    inference_base_model.eval()
    inference_refiner.eval()

    # Normalize if needed
    if lr_image.max() > 2.0:
        lr_image = lr_image / normalization

    lr_tensor = torch.from_numpy(
        lr_image
    ).unsqueeze(0).float().to(device)

    with torch.no_grad():
        # Stage 1: RCAN + HAT
        sr_intermediate = inference_base_model(lr_tensor)

        # Stage 2: MSHAT Refiner
        sr_final, _ = inference_refiner(sr_intermediate)

    return sr_final.squeeze(0).cpu().numpy()


# ----- Example usage -----
print("=" * 55)
print("INFERENCE EXAMPLE")
print("=" * 55)
print("""
# To reload the pipeline later:

base_model, refiner = load_pipeline(
    base_checkpoint_path="rcan_hat_reconstruction_tuned.pth",
    refiner_checkpoint_path="mshat_refiner.pth",
    device="cuda",
)

# Run on a new LR image:
sr_output = run_inference(
    base_model, refiner,
    lr_image,  # (4, 130, 130) numpy array
    device="cuda",
)
# sr_output is (4, 520, 520) at 2.5m resolution
""")
print("=" * 55)
print("PIPELINE COMPLETE")
print("=" * 55)


In [ ]:
# ==========================================================
# CELL 25 — TRAINING CURVES
# ==========================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Total loss ---
ax = axes[0, 0]
ax.plot(
    [h["total"] for h in loss_history],
    color="#2196F3", linewidth=1.5,
)
ax.set_title("Total Loss", fontsize=12)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)

# --- Component losses ---
ax = axes[0, 1]
for key, color, label in [
    ("charb", "#4CAF50", "Charbonnier"),
    ("mse", "#F44336", "MSE"),
    ("ssim", "#FF9800", "1-SSIM"),
]:
    ax.plot(
        [h[key] for h in loss_history],
        color=color, linewidth=1.2, label=label,
    )
ax.set_title("Primary Losses", fontsize=12)
ax.set_xlabel("Epoch")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Validation PSNR ---
if val_history:
    ax = axes[1, 0]
    epochs_v = [v["epoch"] for v in val_history]
    psnrs = [v["psnr"] for v in val_history]
    ax.plot(
        epochs_v, psnrs,
        color="#9C27B0", linewidth=2, marker="o",
    )
    ax.set_title("Validation PSNR (EMA)", fontsize=12)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("PSNR (dB)")
    ax.grid(True, alpha=0.3)

    # --- Validation SSIM ---
    ax = axes[1, 1]
    ssims_v = [v["ssim"] for v in val_history]
    ax.plot(
        epochs_v, ssims_v,
        color="#009688", linewidth=2, marker="s",
    )
    ax.set_title("Validation SSIM (EMA)", fontsize=12)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("SSIM")
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "MSHAT Training Curves",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.savefig(
    "/kaggle/working/mshat_training_curves.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

print("Training curves saved.")
print("=" * 55)


In [ ]:
# ==========================================================
# OPTIONAL CELL — JOINT FINE-TUNING
# ==========================================================
# WARNING: Only run this AFTER the refiner converges.
# Unfreeze RCAN+HAT with a very small LR and fine-tune
# the full pipeline end-to-end.
#
# Risk: Can destabilize. Only do if Phase 2 results are
# already close to targets.
# ==========================================================

RUN_JOINT_FINETUNING = False  # Set to True to enable

if RUN_JOINT_FINETUNING:

    print("=" * 55)
    print("JOINT FINE-TUNING: RCAN+HAT + MSHAT")
    print("=" * 55)

    # Move base model back to GPU
    base_model = base_model.to(device)

    # Unfreeze base model
    for param in base_model.parameters():
        param.requires_grad = True

    # Different learning rates
    joint_optimizer = torch.optim.AdamW([
        {
            "params": base_model.parameters(),
            "lr": 5e-6,  # Very small for base model
        },
        {
            "params": refiner.parameters(),
            "lr": 5e-5,  # 10x for refiner
        },
    ], weight_decay=1e-4)

    joint_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        joint_optimizer,
        T_max=15,
        eta_min=1e-7,
    )

    JOINT_EPOCHS = 15
    JOINT_PATCHES = 600

    joint_scaler = GradScaler()

    best_joint_psnr = best_val_psnr
    best_joint_state_base = None
    best_joint_state_refiner = None

    for epoch in range(JOINT_EPOCHS):

        base_model.train()
        refiner.train()

        total_loss = 0.0
        start_time = time.time()

        progress = tqdm(
            range(JOINT_PATCHES),
            desc=f"Joint {epoch+1}/{JOINT_EPOCHS}",
            unit="patch",
        )

        for step in progress:

            idx = np.random.randint(0, len(train_pairs))
            lr, hr = train_pairs[idx]

            # Extract aligned LR and HR patches
            # LR patch: 48x48, HR patch: 192x192
            lr_h, lr_w = lr.shape[1], lr.shape[2]
            hr_h, hr_w = hr.shape[1], hr.shape[2]

            scale = 4
            lr_patch_size = PATCH_SIZE // scale  # 48

            lr_top = np.random.randint(
                0, lr_h - lr_patch_size + 1
            )
            lr_left = np.random.randint(
                0, lr_w - lr_patch_size + 1
            )

            hr_top = lr_top * scale
            hr_left = lr_left * scale

            lr_patch = lr[
                :,
                lr_top:lr_top + lr_patch_size,
                lr_left:lr_left + lr_patch_size,
            ].copy()

            hr_patch = hr[
                :,
                hr_top:hr_top + PATCH_SIZE,
                hr_left:hr_left + PATCH_SIZE,
            ].copy()

            # Augmentation
            if random.random() > 0.5:
                lr_patch = lr_patch[:, :, ::-1].copy()
                hr_patch = hr_patch[:, :, ::-1].copy()
            if random.random() > 0.5:
                lr_patch = lr_patch[:, ::-1, :].copy()
                hr_patch = hr_patch[:, ::-1, :].copy()

            lr_tensor = torch.from_numpy(
                lr_patch
            ).unsqueeze(0).float().to(device)

            hr_tensor = torch.from_numpy(
                hr_patch
            ).unsqueeze(0).float().to(device)

            # Forward: full pipeline
            joint_optimizer.zero_grad(set_to_none=True)

            with autocast():
                sr_base = base_model(lr_tensor)
                refined, correction = refiner(sr_base)
                loss, _ = criterion(
                    refined, hr_tensor, correction
                )

            joint_scaler.scale(loss).backward()
            joint_scaler.unscale_(joint_optimizer)

            torch.nn.utils.clip_grad_norm_(
                list(base_model.parameters())
                + list(refiner.parameters()),
                1.0,
            )

            joint_scaler.step(joint_optimizer)
            joint_scaler.update()

            total_loss += loss.item()

            progress.set_postfix(
                loss=f"{total_loss/(step+1):.4f}",
            )

        joint_scheduler.step()

        elapsed = time.time() - start_time
        avg_loss = total_loss / JOINT_PATCHES

        print(f"\nJoint Epoch {epoch+1}: "
              f"loss={avg_loss:.6f}, time={elapsed:.1f}s")

        # Validate every 5 epochs
        if (epoch + 1) % 5 == 0:

            # Need to regenerate cached predictions
            # since base model changed
            temp_sr = cache_base_model_outputs(
                val_pairs, name="JointVal"
            )

            val_metrics = validate(
                refiner, temp_sr, val_pairs, device
            )

            print(f"  Joint Val PSNR: {val_metrics['psnr']:.4f}")
            print(f"  Joint Val SSIM: {val_metrics['ssim']:.4f}")
            print(f"  Joint Val RMSE: {val_metrics['rmse']:.6f}")

            if val_metrics["psnr"] > best_joint_psnr:
                best_joint_psnr = val_metrics["psnr"]
                best_joint_state_base = {
                    k: v.detach().cpu().clone()
                    for k, v in base_model.state_dict().items()
                }
                best_joint_state_refiner = {
                    k: v.detach().cpu().clone()
                    for k, v in refiner.state_dict().items()
                }
                print(f"  * New best joint PSNR!")

    # Restore best joint weights
    if best_joint_state_base is not None:
        base_model.load_state_dict(best_joint_state_base)
        refiner.load_state_dict(best_joint_state_refiner)
        print(f"\nBest joint model restored.")
        print(f"Joint PSNR: {best_joint_psnr:.4f}")

    # Save joint-finetuned models
    torch.save(
        {
            "model_state_dict": base_model.state_dict(),
            "model_name": "RCAN_HAT_JointFinetuned",
        },
        os.path.join(save_dir, "rcan_hat_joint_finetuned.pth"),
    )
    torch.save(
        {
            "model_state_dict": refiner.state_dict(),
            "config": refiner_checkpoint["config"],
            "model_name": "MSHAT_JointFinetuned",
        },
        os.path.join(save_dir, "mshat_refiner_joint.pth"),
    )

    print("Joint fine-tuned models saved.")
    print("=" * 55)

else:
    print("Joint fine-tuning: SKIPPED")
    print("Set RUN_JOINT_FINETUNING = True to enable.")
    print("=" * 55)
